In [ ]:
import polars as pl
import pyfaidx
import numpy as np
from pathlib import Path

In [ ]:
# --- paths ---
DATA_ROOT = Path.home() / "Data"
BED_DIR = DATA_ROOT / "encode_beds"
REF_GENOME = DATA_ROOT / "reference" / "hg19.fa"
WINDOWS_DIR = DATA_ROOT / "extracted_windows

# --- parameters ---
STUDIED_SPAN = 10_000
MAX_PEAKS = 10_000
RANDOM_SEED = 42

# --- sanity checks ---
assert DATA_ROOT.exists(), f"Data root not found: {DATA_ROOT}"
assert REF_GENOME.exists(), f"Reference genome not found: {REF_GENOME}"
assert BED_DIR.exists(), f"BED directory not found: {BED_DIR}"

In [ ]:
def read_bed(bed_path: Path) -> pl.DataFrame:
    """
    Read a gzipped ENCODE narrowPeak file and compute peak centres.
    Uses midpoint of start and end as the binding site proxy.
    Returns a Polars DataFrame with columns: peak_id, chr, start, end, center.
    """
    df = pl.read_csv(
        bed_path,
        separator="\t",
        has_header=False,
        comment_prefix="#",
        new_columns=["chr", "start", "end", "name", "score", "strand",
                     "signal_value", "p_value", "q_value", "peak"],
    )

    df = df.select(["chr", "start", "end"]).with_columns([
        ((pl.col("start") + pl.col("end")) // 2).alias("center"),
        pl.Series("peak_id", [f"peak_{i}" for i in range(len(df))]),
    ])

    return df

In [ ]:
def subsample_peaks(df: pl.DataFrame, max_peaks: int = MAX_PEAKS, random_seed: int = RANDOM_SEED) -> pl.DataFrame:
    """
    Subsample to at most max_peaks rows with a fixed random seed.
    If fewer peaks than max_peaks are present, returns all of them.
    """
    if len(df) > max_peaks:
        print(f"Subsampling {len(df)} peaks to {max_peaks}") # maybe print this to a file?
        df = df.sample(n=max_peaks, seed=random_seed)
    else:
        print(f"Fewer than {max_peaks} peaks present ({len(df)}), keeping all")
    return df